# Modul B · Kapitel 4 — Runtime und API

> 🛠️ **Workshop-Version:** Bearbeite die zwei markierten Aufgaben.

**Lernziel:** Du kannst dieselbe Anwendung sauber mit unterschiedlichen Modellservern verbinden.

Dieses Notebook folgt einem kurzen Pfad: Begriff verstehen → Rechnung oder
Messung durchführen → Ergebnis für eine Deployment-Entscheidung nutzen.
Programmiert werden nur zwei Kernstellen: einen portablen Client und Streaming. Hilfs- und
Visualisierungscode ist bewusst vorgegeben.


## 0 · Setup

Die nächsten Zellen laden den OpenAI-kompatiblen Client und die Runtime-Daten. Für Live-Aufrufe muss ein Modellserver erreichbar sein.


In [ ]:
import sys
from pathlib import Path

# helfer.py liegt neben dem Notebook. Der Suchlauf findet es auch, wenn das
# Arbeitsverzeichnis woanders liegt — etwa in Colab.
for kandidat in [Path.cwd(), Path.cwd() / "04_deployment", *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

try:
    import pandas
except ImportError:
    %pip install -q pandas openai tiktoken
    import pandas

import json
import os
import time
import urllib.request
from collections import Counter
from types import SimpleNamespace

from openai import OpenAI

import helfer
from helfer import GB, MODELL, client, frage_llm, lade_daten, zeige_tabelle

# Die nativen Ollama-Endpunkte liegen neben der OpenAI-Naht, nicht unter /v1.
OLLAMA = helfer.BASIS_URL.replace("/v1", "")

# Das Default-Modell denkt vor der Antwort. Über die OpenAI-Naht kommt dann ein
# leerer `content` zurück — warum, steht in Abschnitt 7. Jeder direkte Aufruf in
# diesem Notebook hängt deshalb diesen Zusatz an. `frage_llm` und
# `messe_anfrage` erledigen das schon selbst.
ZUSATZ = {"reasoning_effort": helfer.REASONING} if helfer.REASONING else {}

# Ein Modell ohne internes Denken, für den Vergleich in Abschnitt 7.
VERGLEICHSMODELL = "llama3.2"

print(f"Modell:       {MODELL}")
print(f"OpenAI-Naht:  {helfer.BASIS_URL}")
print(f"nativ:        {OLLAMA}")
print(f"Zusatz:       {ZUSATZ or '(keiner)'}")
print("Setup fertig ✔")

In [ ]:
# ▶️ Datendatei, Modellserver und der Beispieltext, der überall gebraucht wird
RUNTIME_DATEI = lade_daten("03_runtime")
RUNTIMES = RUNTIME_DATEI["runtimes"]
ZUGRIFFSARTEN = RUNTIME_DATEI["zugriffsarten"]
ZIELE = RUNTIME_DATEI["ziele"]

CVE_TEXT = (
    "CVE-2026-3224 — CVSS 9.1. An unauthenticated attacker can send a crafted SAML "
    "assertion to the /sso/acs endpoint of NorthGate VPN Gateway 7.2 to 7.4 and obtain a "
    "valid administrator session. A public proof of concept exists. Patch 7.4.3 is "
    "available. Affected in our estate: 14 gateways, 3 of them reachable from the "
    "internet."
)
BRIEFING = (f"{CVE_TEXT}\n\n"
            "Write a briefing for a CISO in about 120 words: exposure, exploitability, "
            "and the first mitigation step.")
KURZFRAGE = (f"{CVE_TEXT}\n\n"
             "Name the single most urgent action. Answer with at most six words.")

try:
    VERFUEGBAR = {m.id.split(":")[0] for m in client.models.list().data}
    print(f"Ollama antwortet. Installiert: {', '.join(sorted(VERFUEGBAR))}")
except Exception as fehler:
    VERFUEGBAR = set()
    print(f"Ollama nicht erreichbar ({type(fehler).__name__}) — dieses Notebook braucht es.")

print(f"{len(RUNTIMES)} Runtimes, {len(ZUGRIFFSARTEN)} Zugriffsarten, {len(ZIELE)} Ziele "
      f"(Stand {RUNTIME_DATEI['stand']})")

## 1 · Was eine Runtime leistet

Eine Runtime lädt Gewichte, verwaltet Speicher und stellt eine API bereit. Ollama und LM Studio sind lokal bequem; vLLM ist auf Serverdurchsatz ausgelegt.


In [ ]:
# ▶️ Die vier Runtimes
zeige_tabelle([{
    "Runtime": r["name"],
    "Art": r["art"],
    "Scheduler": "ja" if r["scheduler"] else "nein",
    "Format": r["format"],
    "API": r["api"],
    "Passt für": r["passt_fuer"],
} for r in RUNTIMES])

## 2 · Die stabile Naht

Die Anwendung sollte nur drei Dinge kennen: Basis-URL, API-Key und Modellname. So bleibt der restliche Code unabhängig von der Runtime.


In [ ]:
# ▶️ Ein zweiter Client, nur zum Vorführen — der Client der Notebooks kommt aus
# helfer.py und bleibt unangetastet. Die drei Angaben stehen in den ersten Zeilen.
eigener = OpenAI(
    base_url="http://localhost:11434/v1",   # ← 1 · wo die Runtime hört
    api_key="ollama",                       # ← 2 · Ollama prüft den Wert nicht
    timeout=30.0,
)
EIGENES_MODELL = MODELL                     # ← 3 · der Name gehört der Runtime

antwort = eigener.chat.completions.create(
    model=EIGENES_MODELL,
    messages=[{"role": "user",
               "content": "Answer with one word: is a CVSS score of 9.1 "
                          "low, medium, high or critical?"}],
    max_tokens=80,
    temperature=0.0,
    **ZUSATZ,
)

print((antwort.choices[0].message.content or "").strip())
print()
print(f"id             {antwort.id}")
print(f"model          {antwort.model}")
print(f"finish_reason  {antwort.choices[0].finish_reason}")
print(f"usage          {antwort.usage.prompt_tokens} Prompt- + "
      f"{antwort.usage.completion_tokens} Antwort-Tokens")

In [ ]:
# ▶️ Die drei Ziele aus daten/03_runtime.json
TIMEOUT = 5.0     # Sekunden, bis ein Ziel als nicht erreichbar gilt

for name, z in ZIELE.items():
    quelle = (f"fest {z['schluessel_fest']!r}" if z["schluessel_fest"]
              else f"aus ${z['schluessel_umgebung']}")
    print(f"{name:<8} {z['basis_url']:<34} {z['modell']:<36} Schlüssel {quelle}")

### 🛠️ Aufgabe 1 — Einen portablen Client bauen

Erzeuge aus Basis-URL, API-Key und Modellname einen Client. Der Anwendungscode darunter soll für alle Ziele gleich bleiben.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def baue_client(ziel):
    """Baut den OpenAI-Client für ein benanntes Deployment-Ziel."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: baue_client() implementieren")


In [ ]:
# ✅ Selbsttest — geprüft wird die Struktur, nicht die Erreichbarkeit
FELDER = {"ziel", "basis_url", "modell", "client", "bereit", "bemerkung"}

lokal = baue_client("lokal")
assert set(lokal) == FELDER, f"Andere Schlüssel als erwartet: {set(lokal) ^ FELDER}"
assert lokal["ziel"] == "lokal" and lokal["basis_url"].endswith("/v1")
assert lokal["bereit"] and lokal["client"] is not None, \
    "Ollama hat einen festen Schlüssel — dieses Ziel muss immer bereit sein"
assert lokal["client"].base_url.host in {"localhost", "127.0.0.1"}

intern, extern = baue_client("intern"), baue_client("extern")
for gebaut in (intern, extern):
    assert set(gebaut) == FELDER, f"{gebaut['ziel']}: andere Schlüssel als erwartet"
    assert gebaut["bemerkung"], "Jedes Ziel bekommt eine Bemerkung"
    if not gebaut["bereit"]:
        assert gebaut["client"] is None, "Ohne Schlüssel gibt es keinen Client"

assert len({g["modell"] for g in (lokal, intern, extern)}) == 3, "Drei Ziele, drei Modellnamen"

try:
    baue_client("mond")
except KeyError:
    pass
else:
    raise AssertionError("Ein unbekanntes Ziel ist ein Programmierfehler und darf scheitern")

print("✅ Aufgabe 1 gelöst")
print()
zeige_tabelle([{"Ziel": g["ziel"], "base_url": g["basis_url"], "model": g["modell"],
                "bereit": g["bereit"], "Bemerkung": g["bemerkung"]}
               for g in (lokal, intern, extern)])

In [ ]:
# ▶️ Erreichbarkeit prüfen. Ins Internet geht dieses Notebook nicht — das externe
# Ziel wird deshalb nur gebaut, nicht angesprochen.
def pruefe_erreichbar(gebaut):
    """Fragt /v1/models ab und gibt zurück, was dabei herauskommt."""
    if not gebaut["bereit"]:
        return "kein Schlüssel gesetzt"
    if gebaut["basis_url"].startswith("https://"):
        return "nicht geprüft (extern)"
    try:
        namen = {m.id for m in gebaut["client"].models.list().data}
    except Exception as fehler:
        return f"nicht erreichbar ({type(fehler).__name__})"
    gesucht = gebaut["modell"].split(":")[0]
    da = any(n.split(":")[0] == gesucht for n in namen)
    return f"antwortet, {len(namen)} Modelle" + ("" if da else " — das gesuchte fehlt")


zeige_tabelle([{
    "Ziel": name,
    "base_url": ZIELE[name]["basis_url"],
    "Zugriffsart": ZIELE[name]["zugriff"],
    "Status": pruefe_erreichbar(baue_client(name)),
} for name in ZIELE])

## 3 · Standard-API und native API

Die OpenAI-kompatible API macht den Client portabel. Runtime-spezifische Funktionen gehören hinter eine bewusst gewählte Erweiterung.


In [ ]:
# ▶️ /v1/models gegen /api/tags
def nativ(pfad, nutzlast=None):
    """Ruft einen nativen Ollama-Endpunkt auf und gibt das JSON zurück."""
    daten = json.dumps(nutzlast).encode() if nutzlast else None
    anfrage = urllib.request.Request(f"{OLLAMA}{pfad}", data=daten,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(anfrage, timeout=10) as antwort:
        return json.load(antwort)


ueber_naht = client.models.list().data
print(f"/v1/models: {len(ueber_naht)} Einträge, Felder "
      f"{sorted(ueber_naht[0].model_dump())}")
print()
print("/api/tags:")

zeige_tabelle([{
    "Modell": m["name"],
    "auf der Platte (GB)": m["size"] / GB,
    "Parameter": m["details"]["parameter_size"],
    "Quantisierung": m["details"]["quantization_level"],
    "Format": m["details"]["format"],
    "Fähigkeiten": ", ".join(m.get("capabilities", [])),
} for m in nativ("/api/tags")["models"]])

In [ ]:
# ▶️ Was liegt gerade im Speicher? Erst eine Anfrage stellen, dann /api/ps fragen.
frage_llm("Answer with one word: ready?", max_tokens=8)

geladen = nativ("/api/ps")["models"]
if geladen:
    zeige_tabelle([{
        "Modell": m["name"],
        "im Speicher (GB)": m["size"] / GB,
        "davon VRAM (GB)": m["size_vram"] / GB,
        "Kontextfenster": m["context_length"],
        "wird entladen um": m["expires_at"][11:19],
    } for m in geladen])
else:
    print("Gerade ist kein Modell geladen.")

## 4 · Streaming

Streaming verkürzt nicht die Gesamtzeit, aber die wahrgenommene Wartezeit: Die Anwendung zeigt Tokens, sobald sie eintreffen.


### 🛠️ Aufgabe 2 — Streaming messen

Lies den Stream Stück für Stück, sammle den Text und miss Zeit bis zum ersten Token sowie Gesamtdauer.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def stream_messen(prompt, modell=MODELL, max_tokens=250, ausgabe=True):
    """Nimmt die Antwort Stück für Stück entgegen und misst TTFT und Rate."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 2: stream_messen() implementieren")


In [ ]:
# ✅ Selbsttest
lauf = stream_messen(BRIEFING)
print()
print()

assert set(lauf) == {"antwort", "ttft", "sekunden", "stuecke", "tokens_pro_sekunde"}, \
    f"Andere Schlüssel als erwartet: {sorted(lauf)}"
assert lauf["antwort"], "Der Text ist leer — fehlt **ZUSATZ im Aufruf?"
assert lauf["stuecke"] > 20, f"Nur {lauf['stuecke']} Stücke — kam die Antwort in einem Block?"
assert 0 < lauf["ttft"] <= lauf["sekunden"], "TTFT liegt zwischen null und der Gesamtzeit"
assert lauf["tokens_pro_sekunde"] > 1, "Die Rate ist unglaubwürdig klein"

print(f"✅ Aufgabe 2 gelöst — TTFT {lauf['ttft']:.2f} s, {lauf['stuecke']} Stücke, "
      f"{lauf['tokens_pro_sekunde']:.1f} Tokens/s")

In [ ]:
# ▶️ Dieselbe Anfrage mit und ohne Streaming, abwechselnd gemessen
def ohne_stream(prompt, modell=MODELL, max_tokens=250):
    """Dieselbe Anfrage ohne Streaming. Die TTFT ist hier die Gesamtzeit."""
    start = time.perf_counter()
    antwort = client.chat.completions.create(
        model=modell,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
        **ZUSATZ,
    )
    dauer = time.perf_counter() - start
    return {"sekunden": dauer, "ttft": dauer, "tokens": antwort.usage.completion_tokens}


WIEDERHOLUNGEN = 5

# Aufwärmen mit genau diesem Prompt, einmal je Variante.
stream_messen(BRIEFING, ausgabe=False)
ohne_stream(BRIEFING)

# Abwechselnd messen, damit eine Lastspitze nicht nur eine Variante trifft.
mit, ohne = [], []
for _ in range(WIEDERHOLUNGEN):
    mit.append(stream_messen(BRIEFING, ausgabe=False))
    ohne.append(ohne_stream(BRIEFING))

zeige_tabelle([
    {"Variante": "stream=True",
     "TTFT (s)": min(l["ttft"] for l in mit),
     "Gesamtzeit (s)": min(l["sekunden"] for l in mit),
     "Antwort-Tokens": mit[0]["stuecke"]},
    {"Variante": "stream=False",
     "TTFT (s)": min(l["ttft"] for l in ohne),
     "Gesamtzeit (s)": min(l["sekunden"] for l in ohne),
     "Antwort-Tokens": ohne[0]["tokens"]},
])

print(f"Schnellster von {WIEDERHOLUNGEN} Läufen, Modell {MODELL}. Einzelne Läufe, "
      f"TTFT/Gesamtzeit in Sekunden:")
print("  stream=True    " + "   ".join(f"{l['ttft']:.2f}/{l['sekunden']:.2f}" for l in mit))
print("  stream=False   " + "   ".join(f"{l['ttft']:.2f}/{l['sekunden']:.2f}" for l in ohne))

## 5 · Wichtige Parameter

`temperature` steuert Zufälligkeit, `max_tokens` begrenzt die Ausgabe. Für reproduzierbare Messungen halten wir beides konstant.


In [ ]:
# Vorgegebene Hilfsfunktion
def messreihe(prompt, laeufe=8, modell=MODELL, max_tokens=30, **parameter):
    """Stellt dieselbe Frage mehrfach und zählt die verschiedenen Antworten."""
    # Ein ausdrücklich übergebener Parameter gewinnt gegen den Zusatz.
    argumente = {**ZUSATZ, **parameter}

    antworten = []
    for _ in range(laeufe):
        antwort = client.chat.completions.create(
            model=modell,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            **argumente,
        )
        antworten.append((antwort.choices[0].message.content or "").strip())

    haeufigkeit = Counter(a.lower().rstrip(" .!\"'") for a in antworten)
    return {
        "parameter": parameter,
        "laeufe": laeufe,
        "verschieden": len(haeufigkeit),
        "anteil_verschieden": len(haeufigkeit) / laeufe,
        "haeufigste": haeufigkeit.most_common(1)[0][0],
        "antworten": antworten,
    }

## 6 · Der System Prompt

Der System Prompt ist Teil der Anwendungskonfiguration. Änderungen daran können Verhalten und Messwerte ebenso stark verändern wie ein Modellwechsel.


In [ ]:
# ▶️ Dieselbe Frage, einmal ohne und einmal mit System Prompt
SYSTEM = ("You are a SOC analyst. Answer in at most three bullet points. "
          "Every bullet starts with a verb. No preamble and no closing sentence.")
FRAGE = f"{CVE_TEXT}\n\nWhat do we do first?"

print("── ohne System Prompt ──")
print(frage_llm(FRAGE, max_tokens=400))
print()
print("── mit System Prompt ──")
print(frage_llm(FRAGE, system=SYSTEM, max_tokens=400))

In [ ]:
# ▶️ Was im Modell selbst schon hinterlegt ist — /api/show
gezeigt = nativ("/api/show", {"model": MODELL})

print(f"PARAMETER, die in {MODELL} hinterlegt sind:")
print(gezeigt.get("parameters") or "   (keine)")
print()
print("SYSTEM:", repr(gezeigt.get("system")))
print()
print("TEMPLATE, erste acht Zeilen:")
for zeile in (gezeigt.get("template") or "").splitlines()[:8]:
    print(f"   {zeile}")

## 7 · Reasoning-Ausgaben

Einige Modelle trennen Denken und sichtbare Antwort. Der Client muss leeren `content` erkennen und die passende API-Option setzen.


In [ ]:
# ▶️ Derselbe Aufruf, einmal ohne und einmal mit Zusatz
def probiere(modell, **extra):
    """Ein Aufruf mit Zeitmessung, der auch den leeren Fall sauber ausweist."""
    start = time.perf_counter()
    antwort = client.chat.completions.create(
        model=modell,
        messages=[{"role": "user", "content": KURZFRAGE}],
        max_tokens=300, temperature=0.0, **extra,
    )
    wahl = antwort.choices[0]
    # Ollama legt die Gedankenkette in ein eigenes Feld, nicht in content.
    denken = getattr(wahl.message, "reasoning", None) or ""
    return {
        "Modell": modell,
        "Aufruf": "reasoning_effort='none'" if extra else "ohne Zusatz",
        "Sekunden": time.perf_counter() - start,
        "Antwort-Tokens": antwort.usage.completion_tokens,
        "finish_reason": wahl.finish_reason,
        "Zeichen in content": len((wahl.message.content or "").strip()),
        "Zeichen im Denken": len(denken.strip()),
    }


zeilen = [probiere(MODELL), probiere(MODELL, reasoning_effort="none")]
if VERGLEICHSMODELL.split(":")[0] in VERFUEGBAR:
    zeilen.append(probiere(VERGLEICHSMODELL))

zeige_tabelle(zeilen)
print("Gemessen auf diesem Rechner mit max_tokens=300.")

In [ ]:
# Vorgegebene Hilfsfunktion
def frage_robust(prompt, modell=MODELL, max_tokens=300, temperature=0.0):
    """Fragt ein Modell und wiederholt ohne Reasoning, wenn der content leer bleibt."""
    def einmal(**extra):
        antwort = client.chat.completions.create(
            model=modell,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
            **extra,
        )
        wahl = antwort.choices[0]
        return (wahl.message.content or "").strip(), wahl.finish_reason

    text, grund = einmal()
    if text:
        return {"antwort": text, "versuche": 1, "ohne_reasoning": False,
                "finish_reason": grund}

    # Leerer content bei finish_reason="length" heißt: Das Budget ist ins interne
    # Denken gegangen. Zweiter Versuch, diesmal ohne Denken.
    try:
        text, grund = einmal(reasoning_effort="none")
    except Exception as fehler:
        return {"antwort": "", "versuche": 2, "ohne_reasoning": True,
                "finish_reason": f"Fehler: {type(fehler).__name__}"}

    return {"antwort": text, "versuche": 2, "ohne_reasoning": True, "finish_reason": grund}

## 8 · Governance liegt an mehreren Stellen

Authentifizierung, Limits und Logging können im Client, Gateway, Modellserver oder Provider liegen. Zuständigkeiten müssen explizit sein.


In [ ]:
# ▶️ Die vier Zugriffsarten
zeige_tabelle([{
    "Zugriffsart": z["name"],
    "Wo": z["wo"],
    "Netzwerk": z["netzwerk"],
    "Authentifizierung": z["authentifizierung"],
    "Kontingent": z["kontingent"],
    "Logging": z["logging"],
} for z in ZUGRIFFSARTEN])

print()
for z in ZUGRIFFSARTEN:
    print(f"{z['name']:<14} Grenze: {z['grenze']}")

In [ ]:
# ▶️ Die Attrappe für den Selbsttest — sie zählt Aufrufe und scheitert auf Ansage
class Attrappe:
    """Ein Client in der Form von `OpenAI`, der die ersten `fehler` Aufrufe scheitern lässt."""

    def __init__(self, fehler=0, inhalt="ok"):
        self.rest = fehler
        self.inhalt = inhalt
        self.gesehen = []                                # jeder Aufruf mit seinen Argumenten
        self.chat = SimpleNamespace(completions=self)    # client.chat.completions.create(...)

    def create(self, **parameter):
        self.gesehen.append(parameter)
        if self.rest > 0:
            self.rest -= 1
            raise ConnectionError("Server antwortet nicht")
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=self.inhalt))],
            usage=SimpleNamespace(prompt_tokens=11, completion_tokens=7),
        )


probe = Attrappe(fehler=1)
try:
    probe.chat.completions.create(model="attrappe")
except ConnectionError as fehler:
    print(f"erster Aufruf:  {type(fehler).__name__}: {fehler}")
print(f"zweiter Aufruf: {probe.chat.completions.create(model='attrappe').choices[0].message.content}")
print(f"gesehen:        {probe.gesehen}")

In [ ]:
# Vorgegebene Hilfsfunktion
class LLMClient:
    """Eine dünne Schicht um die OpenAI-API: Retry, Timeout, Tokenzählung."""

    def __init__(self, client, modell, versuche=3, timeout=30.0, pause=0.5):
        self.client = client
        self.modell = modell
        self.versuche = versuche
        self.timeout = timeout
        self.pause = pause
        self.aufrufe = 0
        self.fehlversuche = 0
        self.prompt_tokens = 0
        self.antwort_tokens = 0

    def frage(self, prompt, **parameter):
        """Eine Anfrage, bei Fehlern bis zu `versuche` mal wiederholt."""
        letzter = None
        for versuch in range(self.versuche):
            try:
                antwort = self.client.chat.completions.create(
                    model=self.modell,
                    messages=[{"role": "user", "content": prompt}],
                    timeout=self.timeout,
                    **parameter,
                )
            except Exception as fehler:
                self.fehlversuche += 1
                letzter = fehler
                if versuch < self.versuche - 1:
                    time.sleep(self.pause * 2 ** versuch)   # Backoff
                continue

            self.aufrufe += 1
            verbrauch = getattr(antwort, "usage", None)
            if verbrauch is not None:
                self.prompt_tokens += verbrauch.prompt_tokens
                self.antwort_tokens += verbrauch.completion_tokens
            return (antwort.choices[0].message.content or "").strip()

        raise RuntimeError(
            f"{self.versuche} Versuche gegen {self.modell} gescheitert: {letzter}")

    def bericht(self):
        """Was diese Schicht bisher gesehen hat."""
        return {
            "aufrufe": self.aufrufe,
            "fehlversuche": self.fehlversuche,
            "prompt_tokens": self.prompt_tokens,
            "antwort_tokens": self.antwort_tokens,
            "tokens_gesamt": self.prompt_tokens + self.antwort_tokens,
        }

## Fazit

Du kannst eine Anwendung über eine kleine Konfiguration an verschiedene Runtimes anbinden und Antworten streamen. Merksatz: **Portabilität entsteht an der API-Grenze.**
